# Tabulens Tutorial

Este notebook muestra el uso completo de **Tabulens**, una librería de Python para trabajar con datos tabulares mediante un flujo estructurado de:

1. profiling
2. cleaning
3. validation
4. filtering
5. optimization
6. insights

El objetivo es mostrar un caso de uso reproducible y claro sobre un DataFrame con problemas comunes de calidad de datos.

## Instalación

En un entorno reproducible como Google Colab, la librería puede instalarse directamente desde PyPI.

In [ ]:
!pip install tabulens-DavidVarH

## Imports

Se importan las funciones principales de la librería.

In [ ]:
import pandas as pd

from tabulens import (
    RuleSet,
    clean_dataframe,
    generate_insights,
    keep_invalid_rows,
    keep_valid_rows,
    optimize_dataframe,
    profile_dataframe,
    split_valid_invalid_rows,
    validate_dataframe,
)

## Dataset de ejemplo

Se utilizará un dataset sintético que simula información de clientes.

Este dataset contiene algunos problemas típicos de datos reales:

- valores faltantes
- duplicados
- valores fuera de rango
- formatos inválidos
- columnas que pueden mejorarse estructuralmente

In [ ]:
df = pd.DataFrame(
    {
        "cliente_id": [101, 102, 102, 104, None, 106],
        "edad": [25, None, 130, 40, 35, None],
        "estado": ["CDMX", "Jalisco", "Jalisco", None, "CDMX", "Puebla"],
        "email": [
            "ana@email.com",
            "bad_email",
            "bad_email",
            "luis@email.com",
            None,
            "sofia@email.com",
        ],
        "fecha_registro": [
            "2024-01-01",
            "2024-01-02",
            "2024-01-02",
            "2024-02-10",
            None,
            "2024-03-01",
        ],
        "segmento": ["A", "A", "A", "A", "A", "A"],
        "ventas": [100.0, 150.0, 150.0, 300.0, 120.0, 500.0],
    }
)

df

## 1. Profiling

En esta etapa se analiza la estructura del DataFrame para obtener un diagnóstico general de los datos y recomendaciones estructurales.

In [ ]:
profile = profile_dataframe(df)

profile.summary()

In [ ]:
profile.structural_recommendations

### Interpretación

El profiling permite identificar:

- columnas constantes
- posibles columnas de fecha
- posibles recomendaciones estructurales útiles para optimización posterior

Esta etapa no modifica los datos; solo describe su estructura.

## 2. Cleaning

Ahora se aplican estrategias explícitas para manejar valores nulos y duplicados.

En este ejemplo:
- `edad` se rellena con la mediana
- `estado` se rellena con un valor fijo
- `fecha_registro` se rellena con backward fill
- los duplicados por `cliente_id` se eliminan conservando la primera aparición

In [ ]:
cleaning = clean_dataframe(
    df,
    null_strategy={
        "edad": "median",
        "estado": {"method": "fill_value", "value": "DESCONOCIDO"},
        "fecha_registro": "bfill",
    },
    duplicate_strategy={"subset": ["cliente_id"], "keep": "first"},
)

cleaning.cleaned_df

In [ ]:
print(cleaning.render_text())

### Interpretación

Después de la limpieza:

- algunos valores faltantes fueron imputados
- una fila duplicada fue eliminada
- el DataFrame queda más consistente para validación posterior

## 3. Validation

Se definen reglas de calidad de datos para evaluar si ciertas columnas cumplen condiciones deseadas.

In [ ]:
rules = (
    RuleSet()
    .not_null("cliente_id")
    .unique("cliente_id")
    .in_range("edad", min_value=18, max_value=99)
    .allowed_values("estado", ["CDMX", "Jalisco", "Puebla", "DESCONOCIDO"])
    .regex("email", r"^[^@]+@[^@]+\.[^@]+$")
)

In [ ]:
validation = validate_dataframe(cleaning.cleaned_df, rules)

print(validation.render_text())

In [ ]:
validation.summary()

### Interpretación

La validación permite ver:

- qué reglas sí se cumplen
- cuáles no se cumplen
- cuántas filas están afectadas
- los índices concretos de las filas problemáticas

Esto facilita tanto el análisis como el filtrado posterior.

## 4. Filtering

A partir del reporte de validación, se pueden separar las filas válidas e inválidas.

In [ ]:
valid_df = keep_valid_rows(cleaning.cleaned_df, validation, include_near_failing=False)
invalid_df = keep_invalid_rows(cleaning.cleaned_df, validation, include_near_failing=False)

valid_df

In [ ]:
invalid_df

In [ ]:
valid_df_2, invalid_df_2 = split_valid_invalid_rows(
    cleaning.cleaned_df,
    validation,
    include_near_failing=False,
)

print("Valid rows:", len(valid_df_2))
print("Invalid rows:", len(invalid_df_2))

### Interpretación

Esta etapa permite conservar únicamente los registros que cumplen las reglas definidas, lo que puede ser útil antes de análisis posteriores o modelado.

## 5. Optimization

Con las filas válidas, se aplican optimizaciones estructurales seleccionadas por el usuario.

En este ejemplo:
- se convierte `fecha_registro` a datetime
- se elimina una columna constante

In [ ]:
profile_valid = profile_dataframe(valid_df)

optimized = optimize_dataframe(
    valid_df,
    profile_valid,
    selected_rule_ids=["convert_to_datetime", "drop_constant_column"],
)

optimized.optimized_df

In [ ]:
optimized.summary()

### Interpretación

La optimización no se aplica automáticamente: el usuario decide qué recomendaciones estructurales utilizar.

Esto hace que el proceso sea más controlado y transparente.

## 6. Insights

Finalmente, se generan insights de contenido sobre el DataFrame resultante.

A diferencia del profiling, aquí el objetivo no es describir la estructura, sino obtener observaciones útiles sobre los valores.

In [ ]:
insights = generate_insights(optimized.optimized_df)

print(insights.render_text())

In [ ]:
insights.summary()

## Conclusión

Tabulens permite construir un flujo completo y reproducible para trabajar con datos tabulares:

- primero describe la estructura de los datos
- luego permite limpiarlos
- después valida reglas de calidad
- filtra los registros válidos
- aplica optimizaciones elegidas
- y finalmente genera observaciones útiles sobre el contenido

Este enfoque ayuda a estandarizar tareas comunes de preparación de datos y a reducir errores en el análisis.